<a href="https://colab.research.google.com/github/ChrisCopeland123/Document_text_Analyzer/blob/main/Project_2_Milestone_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Manning Live Project
### Project 2: Smart Document Q&A System¶

### 2.0 Vector Database and Search Implementation

#### Environment setup

In [ ]:
import os
import json
import numpy as np
import openai
import re
import pickle
from typing import List, Dict, Any, Tuple
from google.colab import userdata

### 2.1 Vector Database and Search Implementaion

In [ ]:
# Document processor class
class DocumentProcessor:
    def __init__(self, model: str = "gpt-3.5-turbo"):
        """Initialize the DocumentProcessor with model configuration."""
        self.model = model
        self.documents: Dict[str, str] = {}
        self.vector_store: Dict[str, List[float]] = {}

        # Initialize OpenAI client
        self.client = openai.OpenAI(api_key=userdata.get('OpenAI'))


    def chunk_text(self, text: str, chunk_size: int = 1000, overlap: int = 200) -> List[str]:
        """Split text into overlapping chunks for better context preservation."""
        if not text:
            return []

        # Split text into paragraphs first to avoid breaking in the middle of paragraphs
        paragraphs = re.split(r'\n\s*\n', text)

        chunks = []
        current_chunk = ""

        for para in paragraphs:
            # If adding this paragraph exceeds chunk size and we already have content
            if len(current_chunk) + len(para) > chunk_size and current_chunk:
                chunks.append(current_chunk.strip())
                # Keep some overlap for context
                current_chunk = current_chunk[-overlap:] if overlap > 0 else ""

            # Add paragraph to current chunk
            current_chunk += para + "\n\n"

        # Add the last chunk if it has content
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        return chunks

    def _get_embedding(self, text: str) -> list:
        """Generate embedding for text using OpenAI's API."""
        if not text:
            return []

        try:
            response = self.client.embeddings.create(
                model="text-embedding-ada-002",
                input=text
            )
            return response.data[0].embedding

        except Exception as e:
            print(f"Error generating embedding: {e}")
            return []

    def _calculate_similarity(self, embedding1: list, embedding2: list) -> float:
        """Calculate cosine similarity between two embeddings."""
        if not embedding1 or not embedding2:
            return 0.0

        # Convert lists to numpy arrays for efficient computation
        vec1 = np.array(embedding1)
        vec2 = np.array(embedding2)

        # Compute cosine similarity
        dot_product = np.dot(vec1, vec2)
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)


        # Avoid division by zero
        if norm1 == 0 or norm2 == 0:
            return 0.0

        return float(dot_product / (norm1 * norm2))

    def add_document(self, doc_id: str, text: str) -> bool:
        """Process and add a document to the system."""
        if not text:
            print(f"Error: Empty text for document {doc_id}")
            return False

        # Store the original document
        self.documents[doc_id] = text

        # Chunk the document
        chunks = self.chunk_text(text)
        print(f"Document {doc_id} split into {len(chunks)} chunks")

        # Process each chunk to generate embeddings
        successful_chunks = 0
        for i, chunk in enumerate(chunks):
            print(f"Processing chunk {i+1}/{len(chunks)}...")
            embedding = self._get_embedding(chunk)
            if embedding:
                successful_chunks += 1
            else:
                print(f"Failed to generate embedding for chunk {i+1}")

        print(f"Successfully processed {successful_chunks}/{len(chunks)} chunks")
        return successful_chunks > 0

    def add_document_from_file(self, file_path: str) -> bool:
        """Read and add a document from a file."""
        text = read_text_file(file_path)

        if text:
            doc_id = os.path.basename(file_path)
            return self.add_document(doc_id, text)
        return False

    def search_similar_chunks(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Search for the most similar chunks to a given query."""
        if not self.vector_store:
            print("No documents have been processed yet.")
            return []

        # Generate embedding for the query
        query_embedding = self._get_embedding(query)

        if not query_embedding:
            print("Error: Failed to generate embedding for query.")
            return []

        # Calculate similarities with all stored embeddings
        results = []
        for doc_id, chunk_id, chunk_embedding, chunk_text in self.vector_store:
            similarity = self._calculate_similarity(query_embedding, chunk_embedding)

            result_dict = {
                "doc_id": doc_id,
                "chunk_id": chunk_id,
                "text": chunk_text,
                "similarity": similarity,
            }
            results.append(result_dict)

        # Sort results by similarity score and return top k
        sorted_results = sorted(results, key=lambda x: x["similarity"], reverse=True)
        return sorted_results[:top_k]

    def save_state(self, file_path: str) -> bool:
        """Save the current state of the document processor."""
        try:
            state = {
                'documents': self.documents,
                'vector_store': self.vector_store
            }

            with open(file_path, 'wb') as f:
                pickle.dump(state, f)

            print(f"State saved to {file_path}")
            print(f"Saved {len(self.documents)} documents and {len(self.vector_store)} chunks")
            return True

        except Exception as e:
            print(f"Error saving state: {e}")
            return False

    def load_state(self, file_path: str) -> bool:
        """Load a saved state."""
        try:
            with open(file_path, 'rb') as f:
                state = pickle.load(f)

            self.documents = state['documents']
            self.vector_store = state['vector_store']

            print(f"State loaded from {file_path}")
            print(f"Loaded {len(self.documents)} documents and {len(self.vector_store)} chunks")
            return True

        except Exception as e:
            print(f"Error loading state: {e}")
            return False

    def display_document_stats(self):
        """Display basic statistics about processed documents in the system."""
        if not self.documents:
            print("No documents have been processed yet.")
            return

        print("\n" + "="*60)
        print("Document Statistics".center(60))
        print("="*60)

        print(f"- Total documents: {len(self.documents)}")
        print(f"- Total chunks: {len(self.vector_store)}")

        # Group chunks by document
        chunks_by_doc = {}
        for doc_id, chunk_id, _, _ in self.vector_store:
            if doc_id not in chunks_by_doc:
                chunks_by_doc[doc_id] = 0
            chunks_by_doc[doc_id] += 1

        print("\nChunks per Document:")
        for doc_id, count in chunks_by_doc.items():
            char_count = len(self.documents.get(doc_id, ""))
            print(f"  - {doc_id}: {count} chunks ({char_count} characters)")


In [ ]:
# Helper function for reading files
def read_text_file(file_path: str) -> str:
    """Read content from a text file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return ""

In [ ]:
# Test function with comprehensive examples
def test_vector_search():
    """Test the vector search functionality with multiple documents."""

    # Create diverse sample documents for testing
    sample_docs = {
        "contract.txt": """
        Service Agreement between TechCorp Inc. and Global Solutions LLC

        Contract Details:
        Contract Value: $500,000
        Duration: 24 months starting January 1, 2024
        Payment Terms: Net 30 days from invoice date

        Scope of Work:
        Software development and maintenance services
        Technical support and consultation
        System integration and testing

        Liability and Limitations:
        Total liability shall not exceed the contract value
        Force majeure events are excluded from liability
        Both parties maintain confidentiality obligations
        """,

        "tech_report.txt": """
        Technical Infrastructure Assessment Report

        Performance Metrics:
        Database Performance: 40% improvement after optimization
        System Response Time: Reduced from 200ms to 120ms
        Server Uptime: 99.9% availability maintained

        Security Implementation:
        AES-256 encryption implemented across all data streams
        Multi-factor authentication deployed for all users
        Regular security audits conducted quarterly

        Capacity Analysis:
        Current Capacity: 10,000 concurrent users supported
        Peak Usage: 8,500 users during business hours
        Recommended Scaling: Additional 5,000 user capacity by Q2
        """,

        "meeting_minutes.txt": """
        Strategic Planning Meeting Minutes
        Date: March 15, 2024
        Attendees: Sarah Johnson, Michael Chen, Lisa Rodriguez

        Agenda Items Discussed:

        1. Q1 Performance Review
        - Revenue increased by 15% compared to last quarter
        - Client satisfaction scores averaged 4.8/5.0
        - Team productivity metrics exceeded targets

        2. Technology Investment
        - AI document processing system implementation approved
        - Budget allocation: $75,000 for initial deployment
        - Timeline: 6-month implementation schedule

        3. Staffing Updates
        - 3 new analysts hired for data science team
        - Training program scheduled for April 2024
        - Performance reviews due by month end
        """
    }

    # Save sample documents
    for filename, content in sample_docs.items():
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(content)

    print("Testing DocumentProcessor with Vector Search...")
    print("=" * 60)

    # Create DocumentProcessor and add documents
    processor = DocumentProcessor()

    # Add documents to processor
    print("\n1. Adding documents to the system:")
    for filename in sample_docs.keys():
        success = processor.add_document_from_file(filename)
        if success:
            print(f"Successfully added {filename}")
        else:
            print(f"Failed to add {filename}")

    # Display system statistics
    print("\n2. System Statistics:")
    processor.display_document_stats()

    # Test search functionality with various queries
    print("\n3. Testing Search Functionality:")
    test_queries = [
        "What is the contract value?",
        "How much did database performance improve?",
        "Who attended the meeting?",
        "What security measures are implemented?",
        "When is the next review scheduled?"
    ]

    for i, query in enumerate(test_queries, 1):
        print(f"\nQuery {i}: {query}")
        print("-" * 50)

        results = processor.search_similar_chunks(query, top_k=2)

        if results:
            for j, result in enumerate(results, 1):
                print(f"Result {j} (Similarity: {result['similarity']:.4f}):")
                print(f"Source: {result['doc_id']}")
                text_preview = result['text'][:200] + "..." if len(result['text']) > 200 else result['text']
                print(f"Text: {text_preview}")
                print()
        else:
            print("No results found")

    # Test state persistence
    print("\n4. Testing State Persistence:")
    save_success = processor.save_state("vector_db_state.pkl")

    if save_success:
        print("Creating new processor and loading saved state...")
        new_processor = DocumentProcessor()
        load_success = new_processor.load_state("vector_db_state.pkl")

        if load_success:
            print("Testing loaded processor with sample query...")
            test_results = new_processor.search_similar_chunks("contract terms", top_k=1)
            if test_results:
                print(f"Loaded processor working correctly!")
                print(f"Sample result similarity: {test_results[0]['similarity']:.4f}")
            else:
                print("Loaded processor failed to return results")
        else:
            print("Failed to load saved state")
    else:
        print("Failed to save state")

    print("\n" + "=" * 60)
    print("Vector search testing completed!")

# Advanced testing function
def test_similarity_calculations():
    """Test similarity calculation accuracy with known examples."""

    processor = DocumentProcessor()

    # Test with known similar and dissimilar texts
    similar_texts = [
        "The contract value is five hundred thousand dollars",
        "Contract amount totals $500,000"
    ]

    dissimilar_texts = [
        "Database performance improved significantly",
        "The weather is sunny today"
    ]

    print("Testing Similarity Calculations:")
    print("-" * 40)

    # Test similar texts
    emb1 = processor._get_embedding(similar_texts[0])
    emb2 = processor._get_embedding(similar_texts[1])

    if emb1 and emb2:
        similarity = processor._calculate_similarity(emb1, emb2)
        print(f"Similar texts similarity: {similarity:.4f}")
        print(f"Text 1: {similar_texts[0]}")
        print(f"Text 2: {similar_texts[1]}")

    # Test dissimilar texts
    emb3 = processor._get_embedding(dissimilar_texts[0])
    emb4 = processor._get_embedding(dissimilar_texts[1])

    if emb3 and emb4:
        similarity = processor._calculate_similarity(emb3, emb4)
        print(f"\nDissimilar texts similarity: {similarity:.4f}")
        print(f"Text 1: {dissimilar_texts[0]}")
        print(f"Text 2: {dissimilar_texts[1]}")

### 2.2 Run comprehensive test on the document processor class

In [ ]:
# Run comprehensive tes on the document processor
if __name__ == "__main__":
    test_vector_search()
    print("\n" + "=" * 60)
    test_similarity_calculations()

Testing DocumentProcessor with Vector Search...

1. Adding documents to the system:
Document contract.txt split into 1 chunks
Processing chunk 1/1...
Successfully processed 1/1 chunks
Successfully added contract.txt
Document tech_report.txt split into 1 chunks
Processing chunk 1/1...
Successfully processed 1/1 chunks
Successfully added tech_report.txt
Document meeting_minutes.txt split into 1 chunks
Processing chunk 1/1...
Successfully processed 1/1 chunks
Successfully added meeting_minutes.txt

2. System Statistics:

                    Document Statistics                     
- Total documents: 3
- Total chunks: 0

Chunks per Document:

3. Testing Search Functionality:

Query 1: What is the contract value?
--------------------------------------------------
No documents have been processed yet.
No results found

Query 2: How much did database performance improve?
--------------------------------------------------
No documents have been processed yet.
No results found

Query 3: Who att